In [1]:
%load_ext autoreload
%autoreload 2
import os
import sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = '1'
import numpy as np
import pandas as pd
from typing import Dict, List
# from dotenv import load_dotenv  # pip install python-dotenv
from dynclim.features.safety import SegmentationConfig, ThresholdConfig, CompositeConfig 
from dynclim.experiments.runner import ExperimentRunner
from dynclim.data import RailDataPipeline
from dynclim.utils import loadDomainsFromYaml
import arrow
# from dynclim.data.database import get_reports_from_db, create_database_engine
# load_dotenv(dotenv_path='/dynClimToolkit/.env')
# load_dotenv()
from pathlib import Path
# If your notebook is in the project root
# project_root = Path("/Users/mrosen44/Documents/dev/DynamicNarrativeClimateToolkit")
project_root = Path("/Users/michaelrosen/Documents/dev/DynamicNarrativeClimateToolkit")



/Users/michaelrosen/Documents/dev/DynamicNarrativeClimateToolkit/.dynClimVenv/lib/python3.9/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2026-04-14 20:17:56.168 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-14 20:17:56.170 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-14 20:17:56.171 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [2]:
# prepare domains and data for feature extraction
domain_definitions = loadDomainsFromYaml(project_root / 'config/domains/safety_domains.yaml')

# Get data for feature extraction
pipeline = RailDataPipeline(
    events_path=project_root / "dynclim/data/raw/rail/Rail_Equipment_Accident_Incident_Data_(Form_54)_20260204.csv",
    output_dir=project_root / "dynclim/data/processed/rail"
)
results = pipeline.run()
report_data = pipeline.get_report_data()  # [(eid, narrative), ...]

2026-04-14 20:18:21,766 - INFO - ============================================================
2026-04-14 20:18:21,766 - INFO - Running Rail Data Pipeline
2026-04-14 20:18:21,767 - INFO - ============================================================


2026-04-14 20:18:21,767 - INFO - Loading events from /Users/michaelrosen/Documents/dev/DynamicNarrativeClimateToolkit/dynclim/data/raw/rail/Rail_Equipment_Accident_Incident_Data_(Form_54)_20260204.csv
2026-04-14 20:18:26,228 - INFO - Loaded 223,691 raw event records
2026-04-14 20:18:26,357 - WARNING - No raw event ID column found, using index
2026-04-14 20:18:26,985 - INFO - Generated UUIDs for 223,691 events
2026-04-14 20:18:27,376 - INFO - Using existing narrative column: 'Narrative'
2026-04-14 20:18:27,637 - INFO - Using existing date column: 'Date'
2026-04-14 20:18:29,629 - INFO - Cleaning: 223,691 -> 143,535 events (80,156 removed)
2026-04-14 20:18:30,505 - INFO - Saved 143,535 records to /Users/michaelrosen/Documents/dev/DynamicNarrativeClimateToolkit/dynclim/data/processed/rail/events.parquet
2026-04-14 20:18:30,506 - INFO - ============================================================
2026-04-14 20:18:30,506 - INFO - Pipeline complete
2026-04-14 20:18:30,506 - INFO - ===========

In [3]:
# Run experiment with checkpointing

runner = ExperimentRunner(
    domain_definitions = domain_definitions,
    use_acceleration=True,
    device_pref='auto',
    use_parallel=False,
    embedding_model_names=[      
                                 
        'all-MiniLM-L6-v2',        # small, from sentence transformer team
        # 'all-mpnet-base-v2',     # medium, from sentence transformer team
        'all-roberta-large-v1',  # large, from sentence transformer team
        # 'BAAI/bge-large-en-v1.5' # very large, and generally better performance, but has different training method
                                   # when compared to above models which vary just on size
        # 'mixedbread-ai/mxbai-embed-large-v1',   # mid-sized
        'BAAI/bge-m3'                           # successor to BAAI model above
        ]
)

# Test with subset of configurations
config_filter = {
    #'segmentation_methods': ['sentence'],
    #'threshold_methods': ['top_k'],
    #'composite_methods': ['attention_weight', 'weighted_average'],
    'sentiment_methods': ['vader','roberta','siebert'] #['roberta', 'distilbert', 'finbert','vader','siebert']
}

results_df = runner.run_experiment(
    report_data, 
    experiment_name = "rail_04-14-2026",
    config_filter=config_filter,
    resume_from_checkpoint=True,
    # save_interval=2,
    chunk_size=None
    # max_workers=10
)

   Registry: dropped 25 constant columns → 9 remaining (constants in registry_constants.json)

🔬 Running experiment 'rail_04-14-2026'
   Total configurations: 135
   Pending configurations: 135
   Reports: 143535
   Embedding models: 3
   Parallel processing: False
   Caching: Enabled (embeddings=True, segments=True)


Configurations:   0%|          | 0/135 [00:00<?, ?it/s]


🔄 Initializing processor:
   Embedding model: BAAI/bge-m3
   Sentiment method: roberta
🚀 Using Apple Silicon MPS acceleration


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
2026-04-14 20:18:44,433 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Initialized Processor:
   Domains: ['LSC', 'SC', 'ST', 'CSP', 'SEH', 'SI', 'SR']
   Total items: 30
   Device: mps
   Caching: embeddings=True, segments=True
   Injected sentiment cache: 0 entries

🚀 Using batch-optimized processing...
   Pre-segmenting all texts...
[seg] SentenceSegmenter backend=spacy, min_length=5, unit=chars
[seg] built segmenter SentenceSegmenter for key=('5a5a91d711baaba23988779567c0b9f2', 'chars')
   Found 294890 unique segments
   Pre-encoding unique segments...
   ✅ Pre-cached 294890 embeddings

🚀 Starting processing...
   Reports: 143535
   Parallel: False
✅ Processing complete!
   Total time: 4779.03 seconds
   Rate: 30.0 reports/second


... (272 outputs left out)

Configurations:  99%|█████████▉| 134/135 [39:00:43<14:32, 872.46s/it, cache_hit=99.1%, time=941.1s]


💾 Checkpoint saved: 134 configurations
   Full results directory: checkpoints/rail_04-14-2026/_cfg

🚀 Using batch-optimized processing...
   Pre-segmenting all texts...
   Found 294890 unique segments
   Pre-encoding unique segments...
   ✅ Pre-cached 294890 embeddings

🚀 Starting processing...
   Reports: 143535
   Parallel: False
✅ Processing complete!
   Total time: 285.06 seconds
   Rate: 503.5 reports/second


Configurations: 100%|██████████| 135/135 [39:05:32<00:00, 1042.46s/it, cache_hit=99.1%, time=288.5s]



💾 Checkpoint saved: 135 configurations
   Full results directory: checkpoints/rail_04-14-2026/_cfg

💾 Checkpoint saved: 135 configurations
   Full results directory: checkpoints/rail_04-14-2026/_cfg

✅ Experiment complete!
   Total time: 140732.5 seconds
   Average time per config: 1042.46 seconds

📊 Final Cache Statistics:

📊 Cache Performance Report
Device: mps

🎯 Hit Rates:
   Segment Embeddings: 99.0%
   Sentiments: 100.0%
   Text Segmentation: 99.0%
   Similarities: 0.0%
   Overall: 99.1%

📈 Reuse Statistics:
   Unique segments: 292,988
   Total processed: 29,465,865
   Reuse factor: 100.57x

💾 Cache Sizes:
   segment_embeddings: 292,988 entries
   sentiments: 292,988 entries
   text_segments: 130,433 entries
   similarities: 0 entries
📄 Results exported to: checkpoints/rail_04-14-2026_results.csv
📄 Results exported to: checkpoints/rail_04-14-2026_results.h5

📁 Full report-level results saved in: checkpoints/rail_04-14-2026/_cfg
   Each configuration's full results available as s